In [1]:
"""
============================================================
ORZYN AI m2.0
Notebook 03 : Repository Intelligence
============================================================

Purpose
-------
Transform GitHub repository data into structured intelligence.
"""

'\n============================================================\nORZYN AI m2.0\nNotebook 03 : Repository Intelligence\n============================================================\n\nPurpose\n-------\nTransform GitHub repository data into structured intelligence.\n'

In [2]:
from pathlib import Path
import sys

BACKEND_DIR = Path.cwd().parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

In [3]:
from orzyn import *

from dataclasses import dataclass, field

In [4]:
REPOSITORY_QUERY = """
query($owner:String!, $name:String!){

repository(owner:$owner,name:$name){

name

description

url

homepageUrl

createdAt

updatedAt

pushedAt

diskUsage

isArchived

isFork

isPrivate

forkCount

stargazerCount

watchers{

totalCount

}

defaultBranchRef{

name

}

licenseInfo{

name

}

repositoryTopics(first:100){

nodes{

topic{

name

}

}

}

languages(

first:20,

orderBy:{

field:SIZE,

direction:DESC

}

){

edges{

size

node{

name

}

}

}

}

}
"""

In [5]:
@dataclass(slots=True)
class RepositoryProfile:

    name:str

    description:str|None

    url:str

    homepage:str|None

    created_at:object

    updated_at:object

    pushed_at:object

    default_branch:str

    license:str|None

    stars:int

    forks:int

    watchers:int

    disk_usage_kb:int

    archived:bool

    private:bool

    fork:bool

    topics:list[str]=field(default_factory=list)

    languages:dict[str,float]=field(default_factory=dict)

In [6]:
def calculate_language_percentages(edges):

    total = sum(

        edge["size"]

        for edge in edges

    )

    if total == 0:

        return {}

    return {

        edge["node"]["name"]:

        round(

            edge["size"]/total*100,

            2

        )

        for edge in edges

    }

In [7]:
def build_repository_profile(data):

    repo = data["repository"]

    return RepositoryProfile(

        name=repo["name"],

        description=repo["description"],

        url=repo["url"],

        homepage=repo["homepageUrl"],

        created_at=parse_datetime(repo["createdAt"]),

        updated_at=parse_datetime(repo["updatedAt"]),

        pushed_at=parse_datetime(repo["pushedAt"]),

        default_branch=repo["defaultBranchRef"]["name"],

        license=repo["licenseInfo"]["name"]
        if repo["licenseInfo"]
        else None,

        stars=repo["stargazerCount"],

        forks=repo["forkCount"],

        watchers=repo["watchers"]["totalCount"],

        disk_usage_kb=repo["diskUsage"],

        archived=repo["isArchived"],

        private=repo["isPrivate"],

        fork=repo["isFork"],

        topics=[

            topic["topic"]["name"]

            for topic in repo["repositoryTopics"]["nodes"]

        ],

        languages=calculate_language_percentages(

            repo["languages"]["edges"]

        )

    )

In [ ]:
repo_config = get_active_repository()

OWNER = repo_config.owner

REPOSITORY = repo_config.repository

In [8]:
repository_data = client.execute(

    REPOSITORY_QUERY,

    {

        "owner":OWNER,

        "name":REPOSITORY

    }

)

In [9]:
profile = build_repository_profile(

    repository_data

)

profile

RepositoryProfile(name='Orzyn-AI-GitHub-Intelligence', description='🤖 Orzyn AI transforms GitHub repositories into actionable engineering intelligence. It analyzes repository health, engineering velocity, contributor concentration, project risk, productivity trends, historical performance, and provides AI-generated insights.', url='https://github.com/kennykrichardson/Orzyn-AI-GitHub-Intelligence', homepage='https://orzyn-ai.onrender.com', created_at=datetime.datetime(2026, 7, 17, 8, 30, 1, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 7, 17, 8, 39, 55, tzinfo=datetime.timezone.utc), pushed_at=datetime.datetime(2026, 7, 17, 8, 39, 52, tzinfo=datetime.timezone.utc), default_branch='main', license='Other', stars=0, forks=0, watchers=0, disk_usage_kb=5550, archived=False, private=False, fork=False, topics=[], languages={'TypeScript': 61.89, 'Python': 36.35, 'CSS': 1.0, 'Mako': 0.45, 'HTML': 0.28, 'JavaScript': 0.03})

In [10]:
profile.languages

{'TypeScript': 61.89,
 'Python': 36.35,
 'CSS': 1.0,
 'Mako': 0.45,
 'HTML': 0.28,
 'JavaScript': 0.03}

In [11]:
profile.topics

[]

In [12]:
profile.stars

0

In [13]:
profile.default_branch

'main'

In [14]:
profile.created_at

datetime.datetime(2026, 7, 17, 8, 30, 1, tzinfo=datetime.timezone.utc)